# Building a Retrieval-Augmented Generation (RAG) Pipeline for Company Policies

This notebook demonstrates the construction of a Retrieval-Augmented Generation (RAG) pipeline. The goal is to enable an Large Language Model (LLM) to answer questions based on a specific PDF document (e.g., company policies) while also leveraging its general knowledge when context is not found within the document.

### Pipeline Steps:

1.  **Setup and Library Installation:** Install necessary Python libraries.
2.  **Create Dummy Company Policy PDF:** Generate a sample PDF document for demonstration.
3.  **Extract Text from PDF:** Read and extract text content from the PDF.
4.  **Text Chunking:** Divide the extracted text into smaller, manageable chunks.
5.  **Generate Embeddings:** Convert text chunks into numerical vector representations.
6.  **Build FAISS Vector Store:** Store the embeddings in a FAISS index for efficient similarity search.
7.  **Connect to Large Language Model (Groq):** Set up the connection to the Groq API.
8.  **Implement Retrieval-Augmented Generation (RAG) Pipeline:** Combine retrieval and generation to answer user queries.

## Step 1: Setup and Library Installation

First, we need to install all the required Python libraries. These libraries include tools for PDF handling, text processing, embeddings, vector storage, and interacting with Large Language Models.

In [129]:
# Install necessary libraries
!pip install -qq sentence-transformers faiss-cpu reportlab pymupdf langchain langchain-groq langchain-text-splitters

Next, we import all the modules that will be used throughout the notebook.

In [130]:
# Import necessary modules
import fitz  # PyMuPDF for PDF text extraction
from reportlab.pdfgen import canvas # For creating PDFs
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from langchain_text_splitters import RecursiveCharacterTextSplitter # For splitting text into chunks
from sentence_transformers import SentenceTransformer # For generating text embeddings
import faiss # For efficient similarity search (vector store)
import numpy as np
from groq import Groq # For connecting to Groq LLMs
import os # For environment variables (API keys)
from google.colab import userdata # For securely accessing Colab secrets

## Step 2: Create Dummy Company Policy PDF

To simulate a real-world scenario, we will create a dummy PDF document containing various company policies. This PDF will serve as our knowledge base for the RAG pipeline. We use `reportlab` to create a structured document.

In [131]:
def create_company_policy_pdf():
    doc = SimpleDocTemplate("company_policy.pdf", pagesize=letter)
    styles = getSampleStyleSheet()
    story = []

    content = [
        ("ABC Technologies Pvt. Ltd. - Employee Handbook", "h1"),

        ("1. Leave Policy", "h2"),
        ("""
Employees are entitled to:
• 20 days Annual Leave per calendar year.
• 10 days Sick Leave.
• 5 days Casual Leave.

Leave requests must be submitted at least 3 working days in advance through the HR portal.
Unused annual leave may be carried forward up to 10 days.
Any leave exceeding entitlement will be considered unpaid unless approved by HR.
""", "body"),

        ("2. Remote Work Policy", "h2"),
        ("""
Employees may work remotely up to 3 days per week with manager approval.
Core working hours are 10:00 AM to 4:00 PM.
Employees must maintain a stable internet connection and remain available on Microsoft Teams during working hours.
Remote work outside the country requires prior approval from HR.
""", "body"),

        ("3. Working Hours & Attendance", "h2"),
        ("""
Standard office hours are Monday to Friday, 9:00 AM to 6:00 PM.
Employees are allowed a one-hour lunch break.
Late arrivals exceeding 15 minutes more than three times in a month may result in disciplinary action.
Attendance is recorded through the biometric attendance system.
""", "body"),

        ("4. Employee Benefits", "h2"),
        ("""
The company provides the following benefits:
• Annual performance bonus.
• Comprehensive medical insurance.
• Provident Fund.
• Paid maternity and paternity leave.
• Professional certification reimbursement.
• Employee wellness and training programs.
""", "body"),

        ("5. Code of Conduct", "h2"),
        ("""
Employees must maintain professionalism, integrity, and respect in the workplace.
Harassment, discrimination, bullying, and workplace violence are strictly prohibited.
Confidential company information must not be shared with unauthorized individuals.
Violations may result in disciplinary action, including termination.
""", "body"),

        ("6. Information Security Policy", "h2"),
        ("""
Employees must use strong passwords and enable multi-factor authentication.
Company devices should not be shared with unauthorized persons.
Sensitive customer or business information must never be stored on personal devices.
Any suspected security incident must be reported to the IT Security team immediately.
""", "body"),

        ("7. HR & Performance Policy", "h2"),
        ("""
Performance reviews are conducted twice a year.
Employees are encouraged to participate in training and development programs.
Complaints or grievances may be submitted confidentially to the HR department.
The company follows an equal opportunity employment policy and promotes diversity and inclusion.
""", "body"),
    ]

    for text, style_type in content:
        if style_type == "h1":
            story.append(Paragraph(text, styles["Title"]))
        elif style_type == "h2":
            story.append(Spacer(1, 12))
            story.append(Paragraph(text, styles["Heading2"]))
        else:
            story.append(Paragraph(text, styles["Normal"]))
        story.append(Spacer(1, 6))

    doc.build(story)

    print("✅ company_policy.pdf created successfully!")
    print("📄 This PDF contains 7 sections:")
    print("  1. Leave Policy")
    print("  2. Remote Work Policy")
    print("  3. Working Hours & Attendance")
    print("  4. Employee Benefits")
    print("  5. Code of Conduct")
    print("  6. Information Security Policy")
    print("  7. HR & Performance Policy")

create_company_policy_pdf()

✅ company_policy.pdf created successfully!
📄 This PDF contains 7 sections:
  1. Leave Policy
  2. Remote Work Policy
  3. Working Hours & Attendance
  4. Employee Benefits
  5. Code of Conduct
  6. Information Security Policy
  7. HR & Performance Policy


## Step 3: Extract Text from PDF

After creating the PDF, the next step is to extract its textual content. We use PyMuPDF (`fitz`) for efficient PDF parsing.

In [132]:
def read_pdf(file_path):
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

pdf_text = read_pdf("company_policy.pdf")
print(f"Total Characters Read: {len(pdf_text)}")

Total Characters Read: 2315


## Step 4: Text Chunking

Large documents need to be broken down into smaller, manageable chunks. This is crucial for the RAG pipeline, as LLMs have token limits and smaller chunks allow for more precise retrieval. We use `RecursiveCharacterTextSplitter` with a specified `chunk_size` and `chunk_overlap`.

In [133]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = text_splitter.split_text(pdf_text)
print(f"Total Chunks Created: {len(chunks)}")
# Display the first chunk to inspect its content
print("\nFirst chunk example:")
print(chunks[0])

Total Chunks Created: 6

First chunk example:
ABC Technologies Pvt. Ltd. - Employee Handbook
1. Leave Policy
Employees are entitled to: • 20 days Annual Leave per calendar year. • 10 days Sick Leave. • 5 days
Casual Leave. Leave requests must be submitted at least 3 working days in advance through the HR
portal. Unused annual leave may be carried forward up to 10 days. Any leave exceeding entitlement
will be considered unpaid unless approved by HR.
2. Remote Work Policy


## Step 5: Generate Embeddings

To perform similarity searches, we need to convert our text chunks into numerical vector representations, known as embeddings. These embeddings capture the semantic meaning of the text. We use a `SentenceTransformer` model for this purpose.

In [134]:
model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = model.encode(chunks)
print("Embeddings Created!")
print(f"Shape of embeddings: {chunk_embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings Created!
Shape of embeddings: (6, 384)


## Step 6: Build FAISS Vector Store

FAISS (Facebook AI Similarity Search) is a library for efficient similarity search and clustering of dense vectors. We will store our generated embeddings in a FAISS index, which will allow us to quickly find the most relevant text chunks given a user query.

In [135]:
dimension = chunk_embeddings.shape
# Initialize a FlatL2 index (L2 distance is Euclidean distance)
index = faiss.IndexFlatL2(dimension[1])
# Add the embeddings to the index
index.add(np.array(chunk_embeddings))

print(f"Data stored in FAISS Vector DB! Number of vectors: {index.ntotal}")

Data stored in FAISS Vector DB! Number of vectors: 6


### Test Retrieval from FAISS

Before integrating with the LLM, let's test if our FAISS index can retrieve relevant chunks for a simple query.

In [136]:
test_query = "Employee Benefits"
test_query_embedding = model.encode([test_query])

k = 3  # Retrieve top 3 similar results
distances, indices = index.search(np.array(test_query_embedding), k)

print(f"Test Query: {test_query}")
print("\nMost similar chunks found:")
for i in range(len(indices[0])):
    print(f"--- Chunk {i+1} (Distance: {distances[0][i]:.4f}) ---")
    print(chunks[indices[0][i]])

Test Query: Employee Benefits

Most similar chunks found:
--- Chunk 1 (Distance: 1.0837) ---
lunch break. Late arrivals exceeding 15 minutes more than three times in a month may result in
disciplinary action. Attendance is recorded through the biometric attendance system.
4. Employee Benefits
The company provides the following benefits: • Annual performance bonus. • Comprehensive medical
insurance. • Provident Fund. • Paid maternity and paternity leave. • Professional certification
reimbursement. • Employee wellness and training programs.
5. Code of Conduct
--- Chunk 2 (Distance: 1.2234) ---
department. The company follows an equal opportunity employment policy and promotes diversity and
inclusion.
--- Chunk 3 (Distance: 1.2918) ---
should not be shared with unauthorized persons. Sensitive customer or business information must
never be stored on personal devices. Any suspected security incident must be reported to the IT
Security team immediately.
7. HR & Performance Policy
Performance

## Step 7: Connect to Large Language Model (Groq)

We will use the Groq API to access powerful LLMs for generating answers. You need an API key from Groq. It's recommended to store your API key securely in Colab secrets rather than directly in the code.

In [137]:
# Fetch Groq API Key from Colab secrets
# Ensure you have added 'GROQ_API_KEY' to Colab secrets under the '🔑' icon in the left panel.
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

client = Groq()
print("Groq client initialized successfully!")

Groq client initialized successfully!


## Step 8: Implement Retrieval-Augmented Generation (RAG) Pipeline

This is the core of the RAG system. When a user asks a question:

1.  The question is embedded.
2.  The FAISS index is queried to find the most relevant text chunks from the document.
3.  These relevant chunks (context) are then fed to the LLM along with the original question.
4.  The LLM generates an answer based on the provided context. If the context is insufficient, it's instructed to use its general knowledge or state that it doesn't know.

In [138]:
def rag_query(user_question, llm_model="llama-3.3-70b-versatile", num_retrievals=2):
    # 1. Embed the user question
    query_embedding = model.encode([user_question])

    # 2. Search FAISS for relevant chunks
    distances, indices = index.search(np.array(query_embedding), k=num_retrievals)

    # Extract valid indices and retrieve corresponding chunks
    relevant_chunks = [chunks[i] for i in indices[0] if i != -1]

    # 3. Construct the prompt for the LLM
    context = "\n".join(relevant_chunks)
    prompt = f"""Context: {context}
Question: {user_question}

Answer the question based on the provided context. If the information is not found in the context, please use your general knowledge to answer. If you cannot answer from either, state that you don't know."""

    # 4. Query the LLM
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model=llm_model
    )
    return response.choices[0].message.content


# --- Example Usage ---
print("**RAG Pipeline in action:**\n")

query1 = "What is the policy on annual leave?"
answer1 = rag_query(query1)
print(f"Question: {query1}")
print(f"Answer: {answer1}\n")

query2 = "What is social media policy?"
answer2 = rag_query(query2)
print(f"Question: {query2}")
print(f"Answer: {answer2}\n")

query3 = "Who is the current CEO of ABC Technologies?"
answer3 = rag_query(query3)
print(f"Question: {query3}")
print(f"Answer: {answer3}\n")

**RAG Pipeline in action:**

Question: What is the policy on annual leave?
Answer: According to the provided context, the policy on annual leave is as follows: 

- Employees are entitled to 20 days of Annual Leave per calendar year.
- Leave requests must be submitted at least 3 working days in advance through the HR portal.
- Unused annual leave may be carried forward up to 10 days.
- Any leave exceeding entitlement will be considered unpaid unless approved by HR.

Question: What is social media policy?
Answer: There is no information about the social media policy in the provided context. Therefore, I don't know the social media policy based on this context.

Question: Who is the current CEO of ABC Technologies?
Answer: I don't know. The context provided does not mention the current CEO of ABC Technologies.



## Conclusion

This notebook successfully demonstrates how to build a basic yet functional Retrieval-Augmented Generation (RAG) pipeline. By combining a FAISS vector store for efficient document retrieval with a powerful LLM like Groq, we can create a system that intelligently answers questions based on specific documents and falls back on general knowledge when necessary. This approach is highly valuable for internal knowledge bases, customer support, and various other applications where precise, context-aware information retrieval is crucial.

## Resources

The code for this RAG pipeline is available on GitHub at: [Your GitHub Repository Link Here] (Please replace this with your actual GitHub repository link).

In [139]:
# Step 1: Zaroori Libraries Install aur Import Karna
# Sab se pehle apne environment mein ye tools install aur import karein

# Libraries install karein
!pip install sentence-transformers faiss-cpu reportlab pymupdf langchain langchain-groq langchain-text-splitters

import fitz  # PyMuPDF [2]
from reportlab.pdfgen import canvas # PDF banane ke liye [3]
from langchain_text_splitters import RecursiveCharacterTextSplitter # Chinking ke liye [4]
from sentence_transformers import SentenceTransformer # Embeddings ke liye [5]
import faiss # Vector store [5]
import numpy as np
from groq import Groq # LLM connection [6]

In [140]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

def create_company_policy_pdf():
    doc = SimpleDocTemplate("company_policy.pdf", pagesize=letter)
    styles = getSampleStyleSheet()
    story = []

    content = [
        ("ABC Technologies Pvt. Ltd. - Employee Handbook", "h1"),

        ("1. Leave Policy", "h2"),
        ("""
Employees are entitled to:
• 20 days Annual Leave per calendar year.
• 10 days Sick Leave.
• 5 days Casual Leave.

Leave requests must be submitted at least 3 working days in advance through the HR portal.
Unused annual leave may be carried forward up to 10 days.
Any leave exceeding entitlement will be considered unpaid unless approved by HR.
""", "body"),

        ("2. Remote Work Policy", "h2"),
        ("""
Employees may work remotely up to 3 days per week with manager approval.
Core working hours are 10:00 AM to 4:00 PM.
Employees must maintain a stable internet connection and remain available on Microsoft Teams during working hours.
Remote work outside the country requires prior approval from HR.
""", "body"),

        ("3. Working Hours & Attendance", "h2"),
        ("""
Standard office hours are Monday to Friday, 9:00 AM to 6:00 PM.
Employees are allowed a one-hour lunch break.
Late arrivals exceeding 15 minutes more than three times in a month may result in disciplinary action.
Attendance is recorded through the biometric attendance system.
""", "body"),

        ("4. Employee Benefits", "h2"),
        ("""
The company provides the following benefits:
• Annual performance bonus.
• Comprehensive medical insurance.
• Provident Fund.
• Paid maternity and paternity leave.
• Professional certification reimbursement.
• Employee wellness and training programs.
""", "body"),

        ("5. Code of Conduct", "h2"),
        ("""
Employees must maintain professionalism, integrity, and respect in the workplace.
Harassment, discrimination, bullying, and workplace violence are strictly prohibited.
Confidential company information must not be shared with unauthorized individuals.
Violations may result in disciplinary action, including termination.
""", "body"),

        ("6. Information Security Policy", "h2"),
        ("""
Employees must use strong passwords and enable multi-factor authentication.
Company devices should not be shared with unauthorized persons.
Sensitive customer or business information must never be stored on personal devices.
Any suspected security incident must be reported to the IT Security team immediately.
""", "body"),

        ("7. HR & Performance Policy", "h2"),
        ("""
Performance reviews are conducted twice a year.
Employees are encouraged to participate in training and development programs.
Complaints or grievances may be submitted confidentially to the HR department.
The company follows an equal opportunity employment policy and promotes diversity and inclusion.
""", "body"),
    ]

    for text, style_type in content:
        if style_type == "h1":
            story.append(Paragraph(text, styles["Title"]))
        elif style_type == "h2":
            story.append(Spacer(1, 12))
            story.append(Paragraph(text, styles["Heading2"]))
        else:
            story.append(Paragraph(text, styles["Normal"]))
        story.append(Spacer(1, 6))

    doc.build(story)

    print("✅ company_policy.pdf created successfully!")
    print("📄 This PDF contains 7 sections:")
    print("  1. Leave Policy")
    print("  2. Remote Work Policy")
    print("  3. Working Hours & Attendance")
    print("  4. Employee Benefits")
    print("  5. Code of Conduct")
    print("  6. Information Security Policy")
    print("  7. HR & Performance Policy")

create_company_policy_pdf()

✅ company_policy.pdf created successfully!
📄 This PDF contains 7 sections:
  1. Leave Policy
  2. Remote Work Policy
  3. Working Hours & Attendance
  4. Employee Benefits
  5. Code of Conduct
  6. Information Security Policy
  7. HR & Performance Policy


In [141]:
# Step 2: Dummy PDF Create Karna
# Lecture mein testing ke liye ek "Company Policy" PDF banayi gayi thi
# :
def create_dummy_pdf():
    c = canvas.Canvas("company_policy.pdf")
    text = """
Company Policy Document
1. Leave Policy: Employees get 20 days of annual leave per year.
2. Remote Work: You can work remotely from another country for 30 days per year.
3. Salary: Salaries are paid on the 30th of every month.
"""
    # Create a textobject to handle multiline text
    textobject = c.beginText()
    textobject.setTextOrigin(100, 750) # Set starting position
    # Split text by newlines and add each line
    for line in text.split('\n'):
        textobject.textLine(line)
    c.drawText(textobject)
    c.save()
    print("PDF Created!")

create_dummy_pdf()

PDF Created!


In [142]:
# Step 3: PDF Se Text Extract Karna
# Ab PyMuPDF (Fitz) use kar ke PDF se text read karein
# :
def read_pdf(file_path):
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

pdf_text = read_pdf("company_policy.pdf")
print(f"Total Characters Read: {len(pdf_text)}")

Total Characters Read: 227


In [143]:
# Step 4: Chinking (Text Splitting)
# Baray text ko 500 characters ke chunks mein divide karein aur 50 characters ka overlap rakhein
# :
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = text_splitter.split_text(pdf_text)
print(f"Total Chunks Created: {len(chunks)}")

Total Chunks Created: 1


In [144]:
# Step 5: Embeddings Banana
# Sentence Transformer model use kar ke text ko numbers (vectors) mein badlein
# :
model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = model.encode(chunks)
print("Embeddings Created!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings Created!


In [145]:
# Step 6: Vector Database (FAISS) mein Save Karna
# In vectors ko FAISS mein store karein taake search kiya ja sakay
# :
dimension = chunk_embeddings.shape
index = faiss.IndexFlatL2(dimension[1]) # Pass the embedding dimension (384) from the shape tuple
index.add(np.array(chunk_embeddings))
print("Data stored in FAISS Vector DB!")

Data stored in FAISS Vector DB!


In [146]:
# Step 7: Relevant Chunks Search Karna
# Ab ek query banayein, uski embedding generate karein aur FAISS mein search karein
# :
query = "Employee Benefits"
query_embedding = model.encode([query])

k = 3  # Top 2 similar results
distances, indices = index.search(np.array(query_embedding), k)

print(f"Query: {query}")
print("Most similar chunks:")
for i in range(len(indices[0])):
    print(f"Chunk {i+1}: {chunks[indices[0][i]]}")
    print(f"Distance: {distances[0][i]}")

Query: Employee Benefits
Most similar chunks:
Chunk 1: Company Policy Document
1. Leave Policy: Employees get 20 days of annual leave per year.
2. Remote Work: You can work remotely from another country for 30 days per year.
3. Salary: Salaries are paid on the 30th of every month.
Distance: 1.2982227802276611
Chunk 2: Company Policy Document
1. Leave Policy: Employees get 20 days of annual leave per year.
2. Remote Work: You can work remotely from another country for 30 days per year.
3. Salary: Salaries are paid on the 30th of every month.
Distance: 3.4028234663852886e+38
Chunk 3: Company Policy Document
1. Leave Policy: Employees get 20 days of annual leave per year.
2. Remote Work: You can work remotely from another country for 30 days per year.
3. Salary: Salaries are paid on the 30th of every month.
Distance: 3.4028234663852886e+38


In [147]:
# Step 7: Groq LLM Connection
# Groq ki API Key connect karein (is ke liye aap ko Groq dashboard se key leni hogi)
# :
import os

# Apni API key yahan likhein ya Colab secrets se lein
# Option 1: Direct assignment (replace 'YOUR_API_KEY_HERE' with your actual key)
# os.environ["GROQ_API_KEY"] = "YOUR_API_KEY_HERE"

# Option 2: Use Colab secrets (recommended for security)
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

client = Groq()

In [148]:
# Step 8: RAG Query Pipeline (Final Execution)
# Jab user sawal poochay ga, toh system FAISS se relevant chunks nikal kar LLM ko bhejega
# :
user_query = "What is social media policy?"

# 1. Query ko embed karein [27]
query_embedding = model.encode([user_query])

# 2. FAISS mein search karein [28]
D, I = index.search(np.array(query_embedding), k=2)
# Extract valid indices and retrieve corresponding chunks
relevant_chunks = [chunks[i] for i in I[0] if i != -1]

# 3. LLM ko prompt bhein [26]
context = "\n".join(relevant_chunks)
prompt = f"Context: {context}\nQuestion: {user_query}\nAnswer the question based on the provided context. If the information is not found in the context, please use your general knowledge to answer. If you cannot answer from either, state that you don't know."

response = client.chat.completions.create(
    messages=[{"role": "user", "content": prompt}],
    model="llama-3.3-70b-versatile"
)
print("Answer:", response.choices[0].message.content)

Answer: I don't know. The provided context does not mention a social media policy, and I do not have enough general knowledge about the specific company's social media policy to provide an accurate answer.


## Resources

The code for this RAG pipeline is available on GitHub at: [Your GitHub Repository Link Here]